# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mental health survey dataset using the `mlcroissant` library. The dataset is defined by a Croissant schema and includes multiple record sets with demographic and mental health assessment information.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display metadata overview
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record sets, and for each, show its fields and their `@id`s. This information helps identify which `@id`s to use for programmatic access.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"Record Set: {record_set['@id']}")
    if 'field' in record_set and record_set['field']:
        print(" Fields:")
        for field in record_set['field']:
            print(f"  - {field['@id']} | name: {field.get('name', '')}")
    elif 'column' in record_set and record_set['column']:
        print(" Columns:")
        for col in record_set['column']:
            print(f"  - {col['@id']} | name: {col.get('name', '')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above. For this dataset, we select the main survey record set, which contains responses and the mental health assessment scores.

In [ ]:
# Identify main survey record set by @id
main_record_set_id = None
record_set_ids = []
for rs in record_sets:
    record_set_ids.append(rs['@id'])
    # Find the one likely to contain survey responses
    if 'survey' in rs.get('name', '').lower() or 'mental' in rs.get('name', '').lower():
        main_record_set_id = rs['@id']

# If not found by name, default to the first record set
if not main_record_set_id and record_sets:
    main_record_set_id = record_sets[0]['@id']

dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    if records:
        dataframes[rid] = pd.DataFrame(records)

print("Loaded DataFrames:")
for rid in dataframes:
    print(f"- {rid} : shape={dataframes[rid].shape}")
print("\nColumns in main record set:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—filtering, normalizing, grouping—using column `@id`s.

We select numeric mental health scores (`GAD-7`, `PHQ-9`, or `PSQ`) using their `@id`. This example uses the GAD-7 score column.

In [ ]:
# Choose numeric score field (by @id) from main record set
score_field_id = None
for col in dataframes[main_record_set_id].columns:
    if 'GAD' in col or 'gad' in col or 'gad7' in col:
        score_field_id = col
        break
if not score_field_id:
    # Try for PHQ9
    for col in dataframes[main_record_set_id].columns:
        if 'PHQ' in col or 'phq' in col:
            score_field_id = col
            break
if not score_field_id:
    # Try for PSQ
    for col in dataframes[main_record_set_id].columns:
        if 'PSQ' in col or 'psq' in col:
            score_field_id = col
            break

print(f"Using score field: {score_field_id}")

# Filter records where the score is above a threshold
threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][score_field_id] > threshold]
print(f"Filtered records with {score_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{score_field_id}_normalized"] = (
    filtered_df[score_field_id] - filtered_df[score_field_id].mean()) / filtered_df[score_field_id].std()
print(f"Normalized {score_field_id} for filtered records:")
print(filtered_df[[score_field_id, f"{score_field_id}_normalized"]].head())

# Group by a demographic field (e.g., gender, using its @id)
group_field = None
for col in dataframes[main_record_set_id].columns:
    if 'gender' in col.lower():
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[score_field_id].mean()
    print(f"Grouped average {score_field_id} by {group_field}:")
    print(grouped_df.head())
else:
    print("No group field (gender) found for grouping.")

## 5. Visualization
Visualize score distributions and demographic relationships.

For example, display a histogram of GAD-7 scores and a bar plot for average by gender.

In [ ]:
# Histogram of GAD-7/PHQ-9/PSQ scores
plt.figure(figsize=(8, 5))
dataframes[main_record_set_id][score_field_id].hist(bins=12)
plt.title(f"Distribution of {score_field_id}")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

# Bar plot of average score by gender, if available
if group_field:
    grouped_df = dataframes[main_record_set_id].groupby(group_field)[score_field_id].mean()
    grouped_df.plot(kind='bar', figsize=(8,4))
    plt.title(f"Average {score_field_id} by {group_field}")
    plt.ylabel(f"Mean {score_field_id}")
    plt.xlabel(f"{group_field}")
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, filter, normalize, and visualize a Croissant dataset using the `mlcroissant` library.

Key findings from Kilifi County survey records can be further analyzed for mental health trends, stratified by demographic information. The use of `@id` for referencing all dataset entities ensures clarity and reproducibility for FAIR and AI-ready workflows.

*Notebook generated using mlcroissant and the FAIR² mental health survey Croissant schema.*